In [ ]:
import os
from pathlib import Path


def find_repo_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "downstream_tasks/expression_prediction").is_dir():
            return path
    raise RuntimeError("Run inside the GENA-LM clone or set GENA_HOME")


REPO_ROOT = (
    Path(os.environ["GENA_HOME"]).resolve()
    if "GENA_HOME" in os.environ
    else find_repo_root()
)
BENCHMARK_ROOT = Path(os.environ.get("BENCHMARK_ROOT", REPO_ROOT)).resolve()
TASK_ROOT = Path(os.environ.get("TASK_ROOT", REPO_ROOT)).resolve()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", REPO_ROOT / "data")).resolve()


# GENA-LM: compare all models with ground truth

This notebook loops over models, splits and 14/815 cell benchmarks. For 815 cells, it automatically merges `json812 + json3` when a `json815` file is not present.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = TASK_ROOT
# DATA_ROOT is defined above
TRUE_PATH = DATA_ROOT / 'borzoi_and_v1_human_qnorm_matrix_by_id.csv'
SELECTED_TARGETS = DATA_ROOT / 'selected_targets.csv'
OUTPUT_CSV = ROOT / 'comparison_results/gena_all_models_gt_summary.csv'

sys.path.insert(0, str(REPO_ROOT / "downstream_tasks/expression_prediction/benchmarks/gena_lm_benchmark/scripts"))
from score_ct_specificity import score_predictions

# Add/remove rows here. `folder` is where that model's prediction CSV files are stored.
MODELS = [
    {'checkpoint': 'dev_loss',       'folder': ROOT / 'predictions_results/dev_loss'},
    {'checkpoint': 'all_datasets_2', 'folder': ROOT / 'predictions_results/all_datasets2'},
    {'checkpoint': 'all_datasets',   'folder': ROOT / 'predictions_results/all_datasets'},
    {'checkpoint': 'glioma',         'folder': ROOT / 'predictions_results/glioma'},
    {'checkpoint': 'len_2048',       'folder': ROOT / 'predictions_results/len_2048'},
    {'checkpoint': 'ATAC',           'folder': ROOT / 'predictions_results/ATAC'},
    {'checkpoint': 'mult_loss',      'folder': ROOT / 'predictions_results/mult_loss'},
    {'checkpoint': 'xlarge',         'folder': ROOT / 'predictions_results/xlarge'},
]
SPLITS = ['valid', 'test']
BENCHMARKS = [14, 815]

In [ ]:
def load_true(path):
    df = pd.read_csv(path)
    gene_cols = [c for c in df.columns if str(c).startswith(('ENSG', 'ENSMUSG'))]
    return df.set_index('id')[gene_cols].T.astype(float)

def prediction_path(folder, split, n_cells):
    return folder / f'gena_lm_{split}_json{n_cells}_predictions.csv'

def load_predictions(folder, split, n_cells):
    direct = prediction_path(folder, split, n_cells)
    if direct.exists():
        return pd.read_csv(direct).set_index('gene_id'), str(direct)

    # The 815-cell result can be assembled without rerunning the existing 812 cells.
    if n_cells == 815:
        path_812 = prediction_path(folder, split, 812)
        path_3 = prediction_path(folder, split, 3)
        if path_812.exists() and path_3.exists():
            pred_812 = pd.read_csv(path_812).set_index('gene_id')
            pred_3 = pd.read_csv(path_3).set_index('gene_id')
            if pred_812.index.has_duplicates or pred_3.index.has_duplicates:
                raise ValueError(f'Duplicate gene_id in {path_812} or {path_3}')
            pred = pred_812.join(pred_3, how='inner', validate='one_to_one')
            if pred.shape[1] != 815:
                raise ValueError(f'Expected 815 cell columns, found {pred.shape[1]} for {folder.name} {split}')
            return pred, f'{path_812.name} + {path_3.name}'

    return None, None

def correlations(true_log, pred):
    genes = true_log.index.intersection(pred.index)
    cells = true_log.columns.intersection(pred.columns)
    if len(genes) < 2 or len(cells) < 2:
        raise ValueError(f'Not enough overlap: {len(genes)} genes, {len(cells)} cells')

    true = true_log.loc[genes, cells]
    pred = pred.loc[genes, cells].astype(float)

    # Vectorized Pearson correlations: columns = across genes, rows = across cells.
    x = true.to_numpy(dtype=np.float64)
    y = pred.to_numpy(dtype=np.float64)

    x_col = x - x.mean(axis=0, keepdims=True)
    y_col = y - y.mean(axis=0, keepdims=True)
    col_den = np.sqrt((x_col ** 2).sum(axis=0) * (y_col ** 2).sum(axis=0))
    col_corr = np.divide((x_col * y_col).sum(axis=0), col_den,
                         out=np.full(len(cells), np.nan), where=col_den > 0)
    gene_corr = np.nanmean(col_corr)

    x_row = x - x.mean(axis=1, keepdims=True)
    y_row = y - y.mean(axis=1, keepdims=True)
    row_den = np.sqrt((x_row ** 2).sum(axis=1) * (y_row ** 2).sum(axis=1))
    row_corr = np.divide((x_row * y_row).sum(axis=1), row_den,
                         out=np.full(len(genes), np.nan), where=row_den > 0)
    cell_corr = np.nanmean(row_corr)

    true_aligned = true.reset_index().rename(columns={'index': 'gene_id'})
    pred_aligned = pred.reset_index().rename(columns={'index': 'gene_id'})
    return float(gene_corr), float(cell_corr), true_aligned, pred_aligned, len(genes), len(cells)

In [ ]:
split = "test"
n_cells = "815"


pred, source = load_predictions(TASK_ROOT / "predictions_results/xlarge", "valid", 815)
row = {
    'checkpoint': "xlarge",
    'split': split,
    'n_cells_requested': n_cells,
    'gene_corr': np.nan,
    'cell_corr': np.nan,
    'score': np.nan,
    'genes_evaluated': np.nan,
    'cells_evaluated': np.nan,
    'source': source,
    'status': 'missing prediction file' if pred is None else 'ok',
}
if pred is not None:
    try:
        gene_corr, cell_corr, true_a, pred_a, n_genes, n_common_cells = correlations(true_log, pred)
        score = score_predictions(true_a, pred_a, str(SELECTED_TARGETS), need_log=False).get('deviation_r', np.nan)
        row.update(gene_corr=gene_corr, cell_corr=cell_corr, score=score,
                    genes_evaluated=n_genes, cells_evaluated=n_common_cells)
        print('OK', item['checkpoint'], split, n_cells, 'cells:', n_common_cells)
    except Exception as error:
        row['status'] = f'ERROR: {error}'
        print('ERROR', item['checkpoint'], split, n_cells, error)
else:
    print('MISSING', item['checkpoint'], split, n_cells)

In [ ]:
true_raw = load_true(TRUE_PATH)
true_log = np.log2(true_raw + 1)

rows = []
for item in MODELS:
    for split in SPLITS:
        for n_cells in BENCHMARKS:
            pred, source = load_predictions(item['folder'], split, n_cells)
            row = {
                'checkpoint': item['checkpoint'],
                'split': split,
                'n_cells_requested': n_cells,
                'gene_corr': np.nan,
                'cell_corr': np.nan,
                'score': np.nan,
                'genes_evaluated': np.nan,
                'cells_evaluated': np.nan,
                'source': source,
                'status': 'missing prediction file' if pred is None else 'ok',
            }
            if pred is not None:
                try:
                    gene_corr, cell_corr, true_a, pred_a, n_genes, n_common_cells = correlations(true_log, pred)
                    score = score_predictions(true_a, pred_a, str(SELECTED_TARGETS), need_log=False).get('deviation_r', np.nan)
                    row.update(gene_corr=gene_corr, cell_corr=cell_corr, score=score,
                               genes_evaluated=n_genes, cells_evaluated=n_common_cells)
                    print('OK', item['checkpoint'], split, n_cells, 'cells:', n_common_cells)
                except Exception as error:
                    row['status'] = f'ERROR: {error}'
                    print('ERROR', item['checkpoint'], split, n_cells, error)
            else:
                print('MISSING', item['checkpoint'], split, n_cells)
            rows.append(row)

results_long = pd.DataFrame(rows)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_long.to_csv(OUTPUT_CSV, index=False)
display(results_long)
print('Saved:', OUTPUT_CSV)

In [ ]:
summary = results_long.pivot(
    index="checkpoint",
    columns=["n_cells_requested", "split"],
    values=["gene_corr", "cell_corr", "score"],
)

summary = summary.reorder_levels([1, 2, 0], axis=1)

model_order = [
    "dev_loss",
    "all_datasets_2",
    "all_datasets",
    "glioma",
    "len_2048",
    "ATAC",
    "mult_loss",
    "xlarge",
]

metric_order = ["gene_corr", "cell_corr", "score"]

column_order = pd.MultiIndex.from_product(
    [[14, 815], ["valid", "test"], metric_order],
    names=summary.columns.names,
)

summary = summary.reindex(
    index=model_order,
    columns=column_order,
)

display(summary.round(4))
summary.to_csv("summary.csv")

In [ ]:
# Show only missing or failed combinations. Empty output means everything was calculated.
results_long.loc[results_long['status'] != 'ok', ['checkpoint', 'split', 'n_cells_requested', 'status', 'source']]